# পাঠ 11 - মডেল প্রসঙ্গ প্রোটোকল (MCP)

The **মডেল প্রসঙ্গ প্রোটোকল (MCP)** একটি ওপেন স্ট্যান্ডার্ড যা এজেন্টগুলোকে রানটাইমে ডাইনামিকভাবে টুল, রিসোর্স, এবং ডেটা সোর্সগুলো আবিষ্কার এবং ব্যবহার করার যোগ্য করে। কঠোরভাবে টুলগুলোকে এজেন্টের মধ্যে হার্ডকোড করার পরিবর্তে, MCP এজেন্টগুলোকে বাইরের সার্ভারগুলোর সঙ্গে সংযুক্ত হওয়ার অনুমতি দেয় যা অন-ডিমান্ড সক্ষমতা প্রকাশ করে।

In this lesson, you'll learn:
- MCP কী এবং কেন এটি এজেন্ট সিস্টেমগুলোর জন্য গুরুত্বপূর্ণ
- MCP ক্লায়েন্ট-সার্ভার আর্কিটেকচার কিভাবে কাজ করে
- কীভাবে এমন এজেন্ট তৈরি করবেন যেগুলো MCP-স্টাইল টুল আবিষ্কার ব্যবহার করে


## সেটআপ

**প্রয়োজনীয়তা:**
- Azure AI Foundry প্রকল্প যেখানে একটি মডেল মোতায়েন করা হয়েছে
- প্রমাণীকরণের জন্য `az login` চালান


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## মডেল কনটেক্সট প্রোটোকল (MCP) কী?

MCP AI এজেন্টদের বাহ্যিক টুল এবং ডেটা সোর্স আবিষ্কার ও ইন্টারঅ্যাক্ট করার জন্য একটি স্ট্যান্ডার্ড উপায় নির্ধারণ করে:

- **MCP Server**: স্ট্যান্ডার্ড প্রোটোকলের মাধ্যমে টুল, রিসোর্স, এবং প্রম্পটসমূহ প্রকাশ করে
- **MCP Client**: যে এজেন্ট রানটাইম সার্ভারগুলোর সাথে সংযোগ করে এবং উপলব্ধ সক্ষমতাগুলি আবিষ্কার করে
- **Dynamic Discovery**: এজেন্টদের হার্ডকোড করা টুলের প্রয়োজন নেই — তারা রানটাইমে কী উপলব্ধ তা আবিষ্কার করে

এটি প্রসারযোগ্য এজেন্ট সিস্টেম তৈরি করার ক্ষেত্রে শক্তিশালী, যেখানে নতুন সক্ষমতা এজেন্ট কোড পরিবর্তন ছাড়াই যোগ করা যেতে পারে।


## MCP কিভাবে কাজ করে

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. এজেন্ট (MCP ক্লায়েন্ট) একটি MCP সার্ভারের সাথে সংযুক্ত হয়
2. সার্ভার উপলব্ধ টুলগুলির এবং তাদের স্কিমার একটি তালিকা দিয়ে প্রতিক্রিয়া জানায়
3. এরপর এজেন্ট তার যুক্তি প্রক্রিয়ার সময় আবিষ্কৃত যেকোনো টুল কল করতে পারে
4. ফলাফল একই প্রোটোকলের মাধ্যমে ফিরে আসে


## MCP টুল আবিষ্কার অনুকরণ

যেহেতু একটি বাস্তব MCP সার্ভারের জন্য একটি চলমান সার্ভার প্রসেস প্রয়োজন, আমরা `@tool` ফাংশনগুলো ব্যবহার করে সেই প্যাটার্নটি প্রদর্শন করব যা MCP-সংযুক্ত আবাসন পরিষেবা কী প্রদান করত তা অনুকরণ করে।

প্রোডাকশনে, এগুলো লোকালি সংজ্ঞায়িত করার পরিবর্তে ডাইনামিকভাবে একটি MCP সার্ভার থেকে আবিষ্কৃত হত।


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-শৈলীর সরঞ্জাম ব্যবহার করে একটি এজেন্ট তৈরি করা


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## উৎপাদন পরিবেশে MCP

উৎপাদন পরিবেশে, MCP শক্তিশালী প্যাটার্নগুলো সক্ষম করে:

- **ডাইনামিক টুল আবিষ্কার**: এজেন্টরা MCP সার্ভারের সাথে যুক্ত হয় এবং রানটাইমে টুলগুলি আবিষ্কার করে
- **বিচ্ছিন্ন আর্কিটেকচার**: টুল প্রদানকারী এজেন্ট থেকে স্বাধীনভাবে আপডেট করতে পারে
- **সংগঠন জুড়ে শেয়ারিং**: দলগুলি MCP সার্ভারের মাধ্যমে কার্যকারিতা উন্মুক্ত করতে পারে যা যেকোনো এজেন্ট ব্যবহার করতে পারে
- **Microsoft Agent Framework সমর্থন**: MAF-এ `mcp` ইনটিগ্রেশনের মাধ্যমে বিল্ট-ইন MCP ক্লায়েন্ট সমর্থন রয়েছে

To use a real MCP server with MAF, you would connect via `hosted_mcp_tool()` or the MCP client integration.

**আরও জানুন:**
- [MCP স্পেসিফিকেশন](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework-এর MCP সমর্থন](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## সারাংশ

এই পাঠে আপনি যা শিখেছেন:
- **MCP** হল এজেন্ট এবং টুল প্রদানকারীদের মধ্যে ডাইনামিক টুল আবিষ্কারের জন্য একটি ওপেন স্ট্যান্ডার্ড
- **ক্লায়েন্ট-সার্ভার আর্কিটেকচার** এজেন্টদের রানটাইমে ক্ষমতা আবিষ্কার করতে দেয়
- MCP সক্ষম করে **বর্ধনযোগ্য, বিচ্ছিন্ন এজেন্ট সিস্টেম** যেখানে টুলগুলো কোড পরিবর্তন ছাড়াই যোগ করা যায়
- Microsoft Agent Framework প্রোডাকশনে ব্যবহারের জন্য **অন্তর্নির্মিত MCP সমর্থন** প্রদান করে


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
অস্বীকৃতি:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনুবাদ করা হয়েছে। যদিও আমরা নির্ভুলতার দিকে যত্নশীল, স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসামঞ্জস্যতা থাকতে পারে, দয়া করে এটি মনে রাখুন। মূল নথিটি তার নিজ ভাষায়ই কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদ ব্যবহারের ফলে কোনো ভুলবোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়ী নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
